# NFL pick'em odds

Scrapes the Las Vegas odds table from vegasinsider.com and averages the point spread across books.
Negative spread = favored (the more negative, the bigger the expected win).

The site's HTML is flaky — columns get scrambled, junk like `--4.5` shows up, and the spread /
total / moneyline sections are all stacked into one table — so the parsing below validates every
cell and silently drops whatever doesn't make sense instead of crashing.

In [20]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', None)

In [21]:
url = "https://www.vegasinsider.com/nfl/odds/las-vegas/"
tab = pd.read_html(url)[0]

In [22]:
# All parsing / cleaning functions live in pickem.py (shared with email_picks.py, which
# emails Hannah's Entry A picks on a schedule). Edit them there.
from pickem import *


In [23]:
raw, sources = load_sections(tab)
full = raw.copy()
for s in sources:
    full[s] = raw[s].map(parse_line)

# classify sections by typical magnitude: spreads are small, moneylines are +-100 and up,
# and totals ('o47.5 ...') never parse with parse_line at all
med_abs = full.groupby('section')[sources].apply(lambda d: d.abs().median().median())
spread_sections = med_abs[med_abs < 50].index
ml_sections = med_abs[med_abs >= 100].index
total_sections = med_abs[med_abs.isna()].index

spreads = clean_spreads(full[full['section'].isin(spread_sections)], sources)
spreads['ave_spread'] = spreads[sources].mean(axis=1)
spreads['n_books'] = spreads[sources].notna().sum(axis=1)
spreads[['Team'] + sources + ['ave_spread', 'n_books']].sort_values('ave_spread')

,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,ave_spread,n_books
27,49ers,-13.5,-13.5,-13.5,-13.5,-13.5,-13.5,-13.5,-13.5,-13.5,-13.500000,9
15,Ravens,-8.5,-8.5,-8.5,-8.5,-8.5,-9.0,-8.0,-8.0,-8.5,-8.444444,9
11,Buccaneers,-8.5,-8.5,-8.5,-8.5,-8.5,-8.5,-8.0,-8.0,-8.5,-8.388889,9
31,Rams,-7.5,-7.5,-7.0,-7.0,-7.5,-7.0,-7.0,-7.0,-7.0,-7.166667,9
16,Eagles,-7.0,-7.0,-7.0,-7.0,-7.0,-7.0,-7.0,-7.0,-7.0,-7.000000,9
29,Chiefs,-6.5,-6.5,-6.5,-6.5,-6.5,-6.5,-6.5,-7.0,-6.5,-6.555556,9
19,Chargers,-6.5,-6.5,-6.5,-6.5,-6.5,-6.5,-6.5,-6.5,-6.5,-6.500000,9
1,Bills,-5.5,-5.5,-5.5,-5.5,-5.5,-5.5,-5.5,-5.0,-5.5,-5.444444,9
7,Patriots,-5.5,-5.0,-5.5,-5.0,-4.5,-5.5,-5.0,-5.5,-5.5,-5.222222,9
5,Bears,-4.5,-4.5,-4.5,-4.5,-4.5,-4.5,-4.5,-4.5,-4.5,-4.500000,9


## Moneyline → implied win probability

The moneyline section of the same table is literally the odds a team wins the matchup.
Each book's line is converted to an implied probability first (American odds are
nonlinear and asymmetric around ±100, so averaging them directly is biased), then the
probabilities are averaged across books and the vig removed by normalizing each game's
two probabilities to sum to 1. Swapped-team columns are dropped; genuine book
disagreement on toss-ups is kept.

In [24]:
ml = full[full['section'].isin(ml_sections)].reset_index(drop=True).copy()
raw_ml = raw[raw['section'].isin(ml_sections)].reset_index(drop=True)
for s in sources:
    ml[s] = raw_ml[s].map(parse_ml).map(lambda m: implied_prob(m) if not np.isnan(m) else np.nan)

# a swapped column reads p where consensus is 1-p; only decidable when consensus is
# >= 5 points from even (so |med - (1-med)| >= 0.10)
ml = drop_swapped(ml, sources, mirror=lambda p: 1 - p, min_consensus=0.10, label='moneylines')
ml = ml.dropna(subset=sources, how='all').reset_index(drop=True)
ml['q'] = ml[sources].mean(axis=1)                # vig-included average implied prob
ml['n_books'] = ml[sources].notna().sum(axis=1)
for g in range(0, len(ml) - 1, 2):
    tot = ml.loc[g, 'q'] + ml.loc[g + 1, 'q']
    ml.loc[[g, g + 1], 'win_prob'] = ml.loc[[g, g + 1], 'q'] / tot
ml[['Team'] + sources + ['q', 'win_prob', 'n_books']].round(3).sort_values('win_prob', ascending=False)

,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,q,win_prob,n_books
27,49ers,0.909,0.900,0.900,0.909,0.905,0.905,0.909,0.909,0.900,0.905,0.868,9
11,Buccaneers,0.810,0.818,0.820,0.808,0.821,0.818,0.810,0.820,0.820,0.816,0.781,9
15,Ravens,0.810,0.800,0.792,0.800,0.808,0.810,0.810,0.813,0.792,0.804,0.771,9
31,Rams,0.778,0.778,0.783,0.775,0.773,0.778,0.773,0.783,0.783,0.778,0.745,9
16,Eagles,0.773,0.765,0.756,0.770,0.778,0.789,0.773,0.775,0.756,0.771,0.738,9
29,Chiefs,0.750,0.747,0.744,0.756,0.767,0.750,0.750,0.770,0.744,0.753,0.723,9
19,Chargers,0.750,0.750,0.753,0.756,0.756,0.750,0.750,0.759,0.753,0.753,0.722,9
1,Bills,0.710,0.714,0.714,0.701,0.718,0.706,0.710,0.701,0.714,0.710,0.682,9
7,Patriots,0.710,0.701,0.686,0.701,0.706,0.714,0.701,0.706,0.686,0.701,0.672,9
5,Bears,0.688,0.677,0.664,0.686,0.688,0.692,0.688,0.697,0.664,0.683,0.655,9


## Picks by matchup

Same numbers, but one line per game in the order the site lists them, so it's quick to
walk down the pick'em sheet. Format: `away (spread, win prob) @ home (spread, win prob) -> pick`.

In [25]:
sp = spreads.set_index('Team')['ave_spread']
wp = ml.set_index('Team')['win_prob']

games = full[full['section'].isin(spread_sections)].reset_index(drop=True)
for g in range(0, len(games) - 1, 2):
    away, home = games.loc[g, 'Team'], games.loc[g + 1, 'Team']
    pa, ph = wp.get(away, np.nan), wp.get(home, np.nan)
    if np.isnan(pa) and np.isnan(ph):
        continue                                  # game already final / no data
    pick = home if (ph if not np.isnan(ph) else -1) >= (pa if not np.isnan(pa) else -1) else away
    print(f"{away:>12} ({sp.get(away, np.nan):+5.1f}, {pa:4.0%})  @  "
          f"{home:<12} ({sp.get(home, np.nan):+5.1f}, {ph:4.0%})   ->  {pick}")

       Lions ( +5.4,  32%)  @  Bills        ( -5.4,  68%)   ->  Bills
     Packers ( -3.5,  63%)  @  Jets         ( +3.5,  37%)   ->  Packers
     Vikings ( +4.5,  35%)  @  Bears        ( -4.5,  65%)   ->  Bears
    Steelers ( +5.2,  33%)  @  Patriots     ( -5.2,  67%)   ->  Patriots
    Panthers ( -2.5,  57%)  @  Falcons      ( +2.5,  43%)   ->  Panthers
      Browns ( +8.4,  22%)  @  Buccaneers   ( -8.4,  78%)   ->  Buccaneers
     Bengals ( +2.5,  43%)  @  Texans       ( -2.5,  57%)   ->  Texans
      Saints ( +8.4,  23%)  @  Ravens       ( -8.4,  77%)   ->  Ravens
      Eagles ( -7.0,  74%)  @  Titans       ( +7.0,  26%)   ->  Eagles
     Raiders ( +6.5,  28%)  @  Chargers     ( -6.5,  72%)   ->  Chargers
     Jaguars ( +2.6,  42%)  @  Broncos      ( -2.6,  58%)   ->  Broncos
  Commanders ( +4.2,  35%)  @  Cowboys      ( -4.2,  65%)   ->  Cowboys
    Seahawks ( -3.8,  65%)  @  Cardinals    ( +3.8,  35%)   ->  Seahawks
    Dolphins (+13.5,  13%)  @  49ers        (-13.5,  87%)   ->  

## Two-entry strategy

**Entry A (season entry)**: vegas favorite in every game.
**Entry B (chalk with a fuse)**: same as A, except in toss-ups (|spread| <= `TOSSUP`) take the
side the CBS public is *least* on. Once B falls off season-podium pace, set
`B_MODE = 'hunter'`: B then also flips the single game with the highest
edge = P_vegas(dog wins) × (CBS % on the favorite).

`cbs_pick_percents` is the public pick percentage on the **listed team** from the CBS pool page
(the "% picking" bar). Fill it in by hand until the scrape exists; games left out are
treated as unknown (Entry B falls back to chalk for them).

In [26]:
TOSSUP = 1.5          # |ave_spread| <= this is a toss-up (inclusive; matches league_sim's 0.53 cutoff)
B_MODE = 'fuse'       # 'fuse' (chalk + toss-up rule) or 'hunter' (fuse + one aggressive flip)

# public pick % on the named team, from the CBS pool page. One side per game is enough;
# the other side is inferred as 100 - x.
cbs_pick_percents = {   # Week 3, 2026-09-23
    'Packers': 96,
    'Bills': 98,
    'Panthers': 84,
    'Lions': 95,
    'Texans': 51,
    'Jaguars': 68,
    'Chiefs': 99,
    'Giants': 69,
    'Bengals': 90,
    'Seahawks': 99,
    '49ers': 98,
    'Vikings': 72,
    'Ravens': 65,
    'Saints': 67,
    'Rams': 73,
    'Eagles': 92,
}


In [27]:
games = full[full['section'].isin(spread_sections)].reset_index(drop=True)

# validate the hand-entered CBS dict against this week's team tags
teams = set(games['Team'])
bad = [t for t in cbs_pick_percents if t not in teams]
if bad:
    print(f"WARNING: cbs_pick_percents keys not in this week's teams (typo?): {bad}")
    print(f"         valid tags: {sorted(teams)}")
for g in range(0, len(games) - 1, 2):
    both = [t for t in games.loc[[g, g + 1], 'Team'] if t in cbs_pick_percents]
    if len(both) == 2 and sum(cbs_pick_percents[t] for t in both) != 100:
        print(f"WARNING: both sides entered for {both} and they don't sum to 100")
missing = [f"{games.loc[g, 'Team']}@{games.loc[g + 1, 'Team']}" for g in range(0, len(games) - 1, 2)
           if not any(t in cbs_pick_percents for t in games.loc[[g, g + 1], 'Team'])]
if missing:
    print(f"note: no CBS % for {missing}")
rows = []
for g in range(0, len(games) - 1, 2):
    away, home = games.loc[g, 'Team'], games.loc[g + 1, 'Team']
    pa, ph = wp.get(away, np.nan), wp.get(home, np.nan)
    if np.isnan(pa) or np.isnan(ph):
        continue
    fav, dog = (home, away) if ph >= pa else (away, home)
    p_fav = max(pa, ph)
    # CBS % on the favorite, from whichever side was entered
    if fav in cbs_pick_percents:
        cbs_fav = cbs_pick_percents[fav]
    elif dog in cbs_pick_percents:
        cbs_fav = 100 - cbs_pick_percents[dog]
    else:
        cbs_fav = np.nan
    rows.append({'away': away, 'home': home, 'fav': fav, 'dog': dog,
                 'spread': abs(sp.get(fav, np.nan)), 'p_fav': p_fav,
                 'cbs_fav': cbs_fav, 'tossup': abs(sp.get(fav, 0)) <= TOSSUP,
                 'edge': (1 - p_fav) * cbs_fav / 100})
edge = pd.DataFrame(rows)

# ---- Entry A: pure chalk ----
edge['A'] = edge['fav']

# ---- Entry B: chalk, toss-ups go against the public, optional hunter flip ----
edge['B'] = edge['fav']
tu = edge['tossup'] & edge['cbs_fav'].notna()
edge.loc[tu & (edge['cbs_fav'] > 50), 'B'] = edge.loc[tu, 'dog']
edge['B_note'] = ''
edge.loc[tu & (edge['B'] != edge['A']), 'B_note'] = 'toss-up, fade public'
edge.loc[edge['tossup'] & edge['cbs_fav'].isna(), 'B_note'] = 'toss-up, need CBS %'
if B_MODE == 'hunter':
    cand = edge[(edge['B'] == edge['A']) & edge['edge'].notna()]
    if len(cand):
        i = cand['edge'].idxmax()
        edge.loc[i, 'B'] = edge.loc[i, 'dog']
        edge.loc[i, 'B_note'] = f"hunter flip (edge {edge.loc[i, 'edge']:.2f})"

print(f"Entry B mode: {B_MODE}.  B differs from A in {int((edge['A'] != edge['B']).sum())} game(s).\n")
for _, r in edge.iterrows():
    mark = '  <-- ' + r['B_note'] if r['B_note'] else ''
    print(f"{r['away']:>12} @ {r['home']:<12}  A: {r['A']:<12} B: {r['B']:<12}{mark}")

print("\nFlip candidates, ranked by edge = P(dog) x CBS% on favorite:")
(edge[['away', 'home', 'fav', 'spread', 'p_fav', 'cbs_fav', 'edge', 'tossup']]
     .sort_values('edge', ascending=False, na_position='last')
     .round(3))

Entry B mode: fuse.  B differs from A in 0 game(s).

       Lions @ Bills         A: Bills        B: Bills       
     Packers @ Jets          A: Packers      B: Packers     
     Vikings @ Bears         A: Bears        B: Bears       
    Steelers @ Patriots      A: Patriots     B: Patriots    
    Panthers @ Falcons       A: Panthers     B: Panthers    
      Browns @ Buccaneers    A: Buccaneers   B: Buccaneers  
     Bengals @ Texans        A: Texans       B: Texans      
      Saints @ Ravens        A: Ravens       B: Ravens      
      Eagles @ Titans        A: Eagles       B: Eagles      
     Raiders @ Chargers      A: Chargers     B: Chargers    
     Jaguars @ Broncos       A: Broncos      B: Broncos     
  Commanders @ Cowboys       A: Cowboys      B: Cowboys     
    Seahawks @ Cardinals     A: Seahawks     B: Seahawks    
    Dolphins @ 49ers         A: 49ers        B: 49ers       
       Colts @ Chiefs        A: Chiefs       B: Chiefs      
      Giants @ Rams          A: 

,away,home,fav,spread,p_fav,cbs_fav,edge,tossup
4,Panthers,Falcons,Panthers,2.500,0.568,76,0.328,False
1,Packers,Jets,Packers,3.500,0.626,85,0.318,False
12,Seahawks,Cardinals,Seahawks,3.833,0.647,89,0.314,False
2,Vikings,Bears,Bears,4.500,0.655,89,0.307,False
0,Lions,Bills,Bills,5.444,0.682,91,0.290,False
11,Commanders,Cowboys,Cowboys,4.222,0.649,81,0.285,False
6,Bengals,Texans,Texans,2.500,0.568,63,0.272,False
14,Colts,Chiefs,Chiefs,6.556,0.723,97,0.269,False
8,Eagles,Titans,Eagles,7.000,0.738,99,0.259,False
3,Steelers,Patriots,Patriots,5.222,0.672,75,0.246,False


## Tiebreaker

Over/under total for the last matchup of the week, averaged across books.

In [28]:
tot = raw[raw['section'].isin(total_sections)].reset_index(drop=True).copy()
for s in sources:
    tot[s] = tot[s].map(parse_total)
tot['ave_total'] = tot[sources].mean(axis=1)
tot = tot.dropna(subset=['ave_total'])

# last game listed = last matchup of the week; its over/under rows are a pair
away, home = tot['Team'].iloc[-2], tot['Team'].iloc[-1]
tiebreak = tot['ave_total'].iloc[-2:].mean()
print(f"Tiebreaker: {away} @ {home}, vegas total = {tiebreak:.1f}")
# Pool rule is plain "closest". If A and B have identical picks they can both be in the
# same tiebreak, so straddle the total (~3 each side) to cover the distribution. If the
# picks differ they can't both tie, so each entry just wants the median: bracket the
# vegas total with the integers on either side (also avoids sitting exactly on the
# number vegas-following rivals will write, which would only split the tiebreak).
STRADDLE = 3
same_picks = (edge['A'] == edge['B']).all()
if same_picks:
    tb_A, tb_B = round(tiebreak - STRADDLE), round(tiebreak + STRADDLE)
else:
    tb_A, tb_B = int(np.floor(tiebreak)), int(np.floor(tiebreak)) + 1
print(f"  Entry A: {tb_A}   Entry B: {tb_B}   ({'identical picks, straddling' if same_picks else 'picks differ, bracketing'})")
tot[['Team'] + sources + ['ave_total']].iloc[-2:]

Tiebreaker: Giants @ Rams, vegas total = 48.1
  Entry A: 45   Entry B: 51   (identical picks, straddling)


,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,ave_total
30,Giants,48.5,48.0,48.5,48.0,47.5,48.0,48.0,48.0,48.5,48.111111
31,Rams,48.5,48.0,48.5,48.0,47.5,48.0,48.0,48.0,48.5,48.111111


## Hannah's picks (Entry A)

One team per line, in site order, tiebreaker last. Copy/paste.

In [29]:
print("Hannah's picks:")
print("\n".join(edge['A']))
print(f"tiebreak {tb_A}")

Hannah's picks:
Bills
Packers
Bears
Patriots
Panthers
Buccaneers
Texans
Ravens
Eagles
Chargers
Broncos
Cowboys
Seahawks
49ers
Chiefs
Rams
tiebreak 45


## Sean's picks (Entry B)

In [30]:
print("Sean's picks:")
print("\n".join(edge['B']))
print(f"tiebreak {tb_B}")

Sean's picks:
Bills
Packers
Bears
Patriots
Panthers
Buccaneers
Texans
Ravens
Eagles
Chargers
Broncos
Cowboys
Seahawks
49ers
Chiefs
Rams
tiebreak 51
